# Geracao de Normais Climatologicas CHELSA V2.1 (1991-2020)

**Fluxo:**
1. Le os arquivos mensais locais (`chelsa_sa/`)
2. Calcula a media de cada mes ao longo dos 30 anos (12 bandas por variavel)
3. Salva como GeoTIFF multibanda em `chelsa_normals/`

## 1. Configuração

In [ ]:
import rasterio
import numpy as np
from pathlib import Path

VARIABLES  = ['pr', 'pet', 'tas']
YEARS      = range(1991, 2021)
MONTHS     = range(1, 13)

DATA_DIR   = Path('chelsa_sa')
OUTPUT_DIR = Path('chelsa_normals')
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Computar medias mensais

In [ ]:
def compute_normal(var):
    bands   = []
    profile = None
    for mm in MONTHS:
        files    = [DATA_DIR / var / f'CHELSA_{var}_{mm:02d}_{yyyy}_SA.tif' for yyyy in YEARS]
        existing = [f for f in files if f.exists()]
        if not existing:
            print(f'  AVISO: {var} {mm:02d} sem arquivos')
            continue
        arrays = []
        for f in existing:
            with rasterio.open(f) as src:
                arrays.append(src.read(1).astype(np.float32))
                if profile is None:
                    profile = src.profile.copy()
        bands.append(np.nanmean(arrays, axis=0))
        print(f'  {var} {mm:02d}: media de {len(arrays)} anos OK')
    return np.array(bands), profile

## 3. Salvar GeoTIFF multibanda

In [ ]:
def save_normal(var, bands, profile):
    out_path = OUTPUT_DIR / f'CHELSA_{var}_1991-2020.tif'
    profile.update(count=len(bands), dtype='float32', compress='deflate',
                   tiled=True, blockxsize=512, blockysize=512)
    with rasterio.open(out_path, 'w', **profile) as dst:
        for i, band in enumerate(bands, start=1):
            dst.write(band, i)
            dst.update_tags(i, month=f'{i:02d}', variable=var, period='1991-2020')
    size_mb = out_path.stat().st_size / 1024 / 1024
    print(f'  Salvo: {out_path}  ({size_mb:.1f} MB)')
    return out_path

## 4. Executar para as 3 variaveis

In [ ]:
for var in VARIABLES:
    print('\n' + '='*50)
    print(f'Processando: {var}')
    print('='*50)
    bands, profile = compute_normal(var)
    if len(bands) == 0:
        print(f'  ERRO: sem dados para {var}')
        continue
    save_normal(var, bands, profile)

print('\nConcluido! Arquivos salvos em chelsa_normals/')